In [1]:
import pandas as pd
import os

In [2]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [3]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [4]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [5]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [6]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [7]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [8]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [9]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [10]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [11]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [12]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [13]:
df = df.fillna({
    'release_year': -1
})

In [14]:
df.shape

(49783, 18)

In [15]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [16]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                    10
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                    136
release_year                     0
genres_array                    77
production_countries_array     755
production_companies_array    2370
cast_array                     620
director_array                 151
writers_array                 2423
Name: empty_count, dtype: int64


In [17]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [18]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [19]:
df.to_csv(clean_path, index=False)

## ML

In [20]:
# from sentence_transformers import SentenceTransformer
# from transformers import AutoTokenizer, AutoModel
# import torch
import tensorflow_hub as hub
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix

2025-08-18 14:15:23.603063: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-18 14:15:23.619089: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-18 14:15:23.739521: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-18 14:15:23.850268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755526523.933762    1068 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755526523.95

In [21]:
# def vectorize_list_column_top_n(df, col, top_n=500):
#     # transformer les listes en chaînes
#     texts = df[col].fillna("").apply(lambda x: " ".join(x) if isinstance(x, list) else "")
#     vectorizer = CountVectorizer(max_features=top_n)
#     return vectorizer.fit_transform(texts)

# genres_vec = vectorize_list_column_top_n(df, "genres_array", top_n=20)  # genres peu nombreux
# production_countries_vec = vectorize_list_column_top_n(df, "production_countries_array", top_n=50)  # pays de production
# production_companies_vec = vectorize_list_column_top_n(df, "production_companies_array", top_n=100)  # entreprises de production
# cast_vec = vectorize_list_column_top_n(df, "cast_array", top_n=500)     # top 500 acteurs
# director_vec = vectorize_list_column_top_n(df, "director_array", top_n=200)  # top 200 réalisateurs
# writers_vec = vectorize_list_column_top_n(df, "writers_array", top_n=200)  # top 200 scénaristes

In [22]:
def vectorize_list_column(df, col, top_k=None):
    mlb = MultiLabelBinarizer()
    X = mlb.fit_transform(df[col])
    if top_k:
        # garder uniquement les top_k colonnes les plus fréquentes
        freqs = X.sum(axis=0)
        top_idx = freqs.argsort()[::-1][:top_k]
        X = X[:, top_idx]
    return csr_matrix(X), mlb

In [23]:
genres_vec, genres_mlb = vectorize_list_column(df, 'genres_array', top_k=50)
cast_vec, cast_mlb = vectorize_list_column(df, 'cast_array', top_k=500)
director_vec, director_mlb = vectorize_list_column(df, 'director_array', top_k=200)
production_companies_vec, production_companies_mlb = vectorize_list_column(df, 'production_companies_array', top_k=100)
production_countries_vec, production_countries_mlb = vectorize_list_column(df, 'production_countries_array', top_k=50)
writers_vec, writers_mlb = vectorize_list_column(df, 'writers_array', top_k=200)

In [24]:
# # 1. Fonction de tokenisation rapide avec regex
# def tokenize_and_filter_regex(text):
#     tokens = re.findall(r"[a-zA-Z]+", str(text).lower())  # garde uniquement les mots alphabétiques
#     return [t for t in tokens if t not in ENGLISH_STOP_WORDS]  # enlève les stopwords sklearn

# # 2. Application sur tout le DataFrame
# df["overview_filtered"] = df["overview"].apply(tokenize_and_filter_regex)

In [25]:
# # 3. Transformation en texte pour CountVectorizer
# df["overview_filtered_str"] = df["overview_filtered"].apply(lambda x: " ".join(x))

In [26]:
# # 4. TF
# count_vectorizer = CountVectorizer(max_features=10000)
# overview_tf = count_vectorizer.fit_transform(df["overview_filtered_str"])

In [27]:
# # 5. TF-IDF
# tfidf_transformer = TfidfTransformer()
# overview_tfidf = tfidf_transformer.fit_transform(overview_tf)

In [28]:
# # TF-IDF sur les descriptions
# tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
# overview_tfidf = tfidf_vectorizer.fit_transform(df["overview"].fillna(""))

In [29]:
# model = SentenceTransformer('all-MiniLM-L6-v2')  # petit et rapide
# overview_embeddings = model.encode(df['overview'].fillna("").tolist(), show_progress_bar=True)

In [30]:
# # Choisir un modèle léger
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)

In [31]:
# # Fonction pour obtenir embeddings
# def get_embeddings(texts):
#     inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
#     with torch.no_grad():
#         outputs = model(**inputs)
#     # Moyenne sur la dimension des tokens
#     embeddings = outputs.last_hidden_state.mean(dim=1)
#     return embeddings.numpy()

# # Exemples : embeddings pour les descriptions
# descriptions = df['overview'].tolist()
# embeddings = get_embeddings(descriptions)

In [32]:
# Charger le modèle (en cache la première fois)
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

2025-08-18 14:16:15.598072: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [33]:
descriptions = df['overview'].tolist()
# embeddings = embed(descriptions).numpy()

In [34]:

batch_size = 512  # ou 256 si ça plante encore
embeddings_list = []

for i in range(0, len(descriptions), batch_size):
    batch = descriptions[i:i+batch_size]
    batch_emb = embed(batch).numpy()
    embeddings_list.append(batch_emb)

2025-08-18 14:16:20.322738: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [35]:
embeddings = np.vstack(embeddings_list)

In [36]:
embeddings

array([[ 0.04591452,  0.01079546, -0.06505066, ..., -0.01336516,
         0.06733175,  0.01074905],
       [ 0.0032881 , -0.04805043,  0.04147273, ..., -0.02304581,
         0.04798687,  0.05125323],
       [ 0.02407429, -0.05371161, -0.02921458, ...,  0.05831166,
         0.07037839,  0.06184515],
       ...,
       [-0.03192678,  0.06454503,  0.06910776, ..., -0.04658826,
         0.07801674, -0.06933897],
       [ 0.01836303,  0.04568144,  0.02971495, ..., -0.05616704,
         0.02479967,  0.02216027],
       [ 0.01089489,  0.0718065 ,  0.06596401, ..., -0.00334176,
         0.07496444,  0.06631833]], dtype=float32)

In [37]:
# -------------------------
# Scaling pour release_year
# -------------------------

scaler = StandardScaler()
numeric_features = scaler.fit_transform(df[['release_year']])

In [38]:
from sklearn.preprocessing import normalize

# normaliser en dense (ou sparse, mais en float)
embeddings_norm = normalize(embeddings).astype(np.float32)
genres_norm = normalize(genres_vec, axis=1)
cast_norm = normalize(cast_vec, axis=1)
director_norm = normalize(director_vec, axis=1)
prod_comp_norm = normalize(production_companies_vec, axis=1)
prod_countries_norm = normalize(production_countries_vec, axis=1)
writers_norm = normalize(writers_vec, axis=1)
numeric_norm = normalize(numeric_features, axis=1)

In [39]:
# convertir sparse si nécessaire
genres_norm = csr_matrix(genres_norm)
cast_norm = csr_matrix(cast_norm)
director_norm = csr_matrix(director_norm)
prod_comp_norm = csr_matrix(prod_comp_norm)
prod_countries_norm = csr_matrix(prod_countries_norm)
writers_norm = csr_matrix(writers_norm)
numeric_norm = csr_matrix(numeric_norm)

In [56]:
# -------------------------
# Assemblage sparse
# -------------------------
X = hstack([
    csr_matrix(embeddings_norm).multiply(2.5),
    genres_norm.multiply(1.75),
    cast_norm.multiply(1.25),
    director_norm.multiply(1.0),
    prod_comp_norm.multiply(0.25),
    prod_countries_norm.multiply(0.5),
    writers_norm.multiply(0.75),
    numeric_norm.multiply(0.5)
])
# X = hstack([
#     csr_matrix(embeddings_norm).multiply(1.0),
# ])

In [57]:
k = 10  # nombre de films similaires que tu veux récupérer

In [58]:
# -------------------------
# 5. NearestNeighbors
# -------------------------
nn_model = NearestNeighbors(n_neighbors=k+1, metric='cosine')  # 1 film = lui-même
nn_model.fit(X)

,n_neighbors,11
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [59]:
# -------------------------
# 6. Fonction de recommandation
# -------------------------
def recommend(movie_idx, top_n=10):
    distances, indices = nn_model.kneighbors(X[movie_idx], n_neighbors=top_n+1)
    # ignorer le premier, c'est le film lui-même
    recommended_indices = indices[0][1:]
    return df.iloc[recommended_indices][['title', 'release_year', 'genres_array', 'overview', 'cast_array', 'director_array', 'production_companies_array', 'production_countries_array', 'writers_array']]

In [60]:
test = recommend(424)
test

,title,release_year,genres_array,overview,cast_array,director_array,production_companies_array,production_countries_array,writers_array
121404,Batman v Superman: Dawn of Justice,2016,"[Action, Adventure, Fantasy]",Fearing the actions of a god-like Super Hero l...,"[Tom Whalen, Jean Ho, Tiffany L. Addison, Harr...",[Zack Snyder],"[Warner Bros. Pictures, RatPac Entertainment, ...",[United States of America],"[Chris Terrio, Jay Oliva, Bob Kane, William Mo..."
223345,Mythica: The Darkspore,2015,"[Action, Adventure, Fantasy]",Fighting through creature-infested lands and p...,"[Cole Johnson, Clare Niederpruem, Brogan Johns...",[Anne K. Black],[Arrowstorm Entertainment],[United States of America],"[Jason Faller, Justin Partridge, Kynan Griffin..."
31589,Deathstalker IV: Match of Titans,1991,"[Action, Adventure, Fantasy]",What begins as a grand tournament soon turns s...,"[Mihaela Staikova, Stancho Stanchev, Velico St...",[Howard R. Cohen],[Concorde-New Horizons],[United States of America],[Howard R. Cohen]
242370,Mythica: The Necromancer,2015,"[Fantasy, Action, Adventure]",The young wizard Marek is forced to lead her b...,"[Davey Morrison, James C. Morris, Sahna Foley,...",[A. Todd Smith],"[Camera 40 Productions, Arrowstorm Entertainme...",[United States of America],"[Jason Faller, Justin Partridge, Liska Ostojic]"
98507,Wonder Woman,1974,"[Action, Adventure, Fantasy, TV Movie]",A super-hero uses her powers to thwart an inte...,"[Kaz Garas, Richard X. Slattery, Jordan Rhodes...",[Vincent McEveety],[Warner Bros. Television],[United States of America],[John D. F. Black]
23384,Conan the Barbarian,2011,"[Adventure, Fantasy, Action]",A quest that begins as a personal vendetta for...,"[Katarzyna Wolejnio, Bashar Rahal, Daniel Rash...",[Marcus Nispel],"[Dark Horse Entertainment, Nu Image, Paradox E...",[United States of America],"[Dan Rosenfelt, Thomas Dean Donnelly, Joshua O..."
181489,Wonder Woman,2017,"[Action, Adventure, Fantasy]",An Amazon princess comes to the world of Man i...,"[Camilla Roholm, Freddy Elletson, Caroline Win...",[Patty Jenkins],"[Atlas Entertainment, Cruel & Unusual Films, D...",[United States of America],"[Zack Snyder, Allan Heinberg, William Moulton ..."
561842,Zack Snyder's Justice League,2021,"[Action, Adventure, Fantasy]",Determined to ensure Superman's ultimate sacri...,"[Tina Balthazar, Harry Lennix, Sam Benjamin, H...",[Zack Snyder],"[Warner Bros. Pictures, The Stone Quarry, Atla...",[United States of America],"[Chris Terrio, Marv Wolfman, Will Beall, Bob K..."
204673,In the Lost Lands,2025,"[Action, Fantasy, Adventure]",A queen sends the powerful and feared sorceres...,"[Fraser James, Amara Okereke, Tue Lunding, Dei...",[Paul W. S. Anderson],"[Constantin Film, Spark Productions]","[Germany, Switzerland]","[Paul W. S. Anderson, George R. R. Martin, Con..."
256044,Mythica: The Iron Crown,2016,"[Action, Adventure, Fantasy]",When a team of unlikely heroes hijacks a steam...,"[James Gaisford, Kaza Marie Ayersman, Paul D. ...",[John Lyde],"[Mainstay Productions, Arrowstorm Entertainmen...",[United States of America],[Jason Faller]
